# Alert 생성기 실험 ② — 묶음이 정답(세탁 시도)과 얼마나 같은가

같은 씨앗(모델 점수 ≥ 의심 기준)을 두 가지 방법으로 묶고, 데이터셋의 **세탁 시도**(Patterns 파일의 BEGIN ~ END 한 덩어리)와 비교한다.

| 방법 | 묶는 규칙 |
|---|---|
| G 생성기 | 서비스 Alert 생성기. 01번 결과(`members.parquet`)를 그대로 읽는다 |
| B 단순 연결 | 씨앗끼리 계좌를 공유하면 같은 묶음. 금액·방향·바쁜 계좌는 보지 않는다. 시간 제한만 바꿔 본다 |

- 정답은 시도마다 **평가 구간 안에 있는 거래**만 쓴다(구간 안 거래가 2건 이상인 시도). 씨앗이 평가 구간에만 있어서 구간 밖 거래는 어느 방법도 묶을 수 없다.
- 첫 버전은 "구간 안에서 시작하고 끝난 시도"만 썼는데, HI-Large 에서는 그런 시도가 103개뿐이고 거의 다 거래 2건짜리라 대표성이 없었다(2026-11-22).
- `EXCLUDE` 의 패턴 거래는 묶음과 정답 양쪽에서 뺀다.
- 01번을 먼저 돌려야 한다. 생성기를 다시 돌리지는 않는다.

## 0. 설정

In [1]:
DATASET = 'HI-Small'
RUN = 'before_after_grouping'            # 01번 결과 폴더
EXCLUDE = ['STACK', 'BIPARTITE']         # 평가에서 빼는 패턴
BASELINES = {'B0 계좌 공유': None, 'B1 계좌 공유 + 7일 이내': 168, 'B2 계좌 공유 + 1일 이내': 24}   # 이름: 시간 제한(시간)

In [2]:
from pathlib import Path
import json
import duckdb, numpy as np, pandas as pd
import gen_lib as GL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir() or (p / 'datasets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
RUN_DIR = Path.cwd() / 'results' / DATASET / RUN
OUT = Path.cwd() / 'results' / DATASET / 'grouping_vs_truth'
OUT.mkdir(parents=True, exist_ok=True)
run = json.load(open(RUN_DIR / 'run.json', encoding='utf-8'))
a, b = run['test_window']
print('01번 실행:', run['scores'], '| 기준', run['threshold'], '| 평가 구간', a, '~', b)

FileNotFoundError: [Errno 2] No such file or directory: '/home/donghyeon/workspace/money_laundry_ai_/code/rule_engine/results/HI-Small/before_after_grouping/run.json'

## 1. 01번 결과 읽기
`members.parquet` 에는 씨앗과 Alert 구성 거래가 들어 있다. 계좌와 시각은 거래 파일에서 붙인다.

In [ ]:
members = pd.read_parquet(RUN_DIR / 'members.parquet')
con = duckdb.connect()
con.register('member_ids', members[['edge_id']])
tx = con.execute(f"""SELECT t.edge_id, t.ts, t.src, t.dst
                     FROM read_parquet('{BASE / 'transactions.parquet'}') t JOIN member_ids USING (edge_id)""").df()
members = members.merge(tx, on='edge_id', how='left')
assert members.ts.notna().all(), '거래 파일에 없는 edge_id 가 있다'
print(f'거래 {len(members):,} 건 | 씨앗 {int(members.seed_model.sum()):,} | Alert 구성 거래 {int((members.role > 0).sum()):,}')

## 2. 정답: 세탁 시도
시도마다 평가 구간 안의 거래 수를 세고, 평가에 쓸 시도를 고른다.

In [ ]:
PAT = BASE / 'patterns.parquet'
start, end = pd.Timestamp(a), pd.Timestamp(b) + pd.Timedelta(days=1)
IN = f"ts >= TIMESTAMP '{start}' AND ts < TIMESTAMP '{end}'"
attempts = con.execute(f"""SELECT attempt_id, any_value(pattern_type) AS pattern_type, count(*) AS n_rows,
                                  count(*) FILTER (WHERE {IN}) AS n_in, count(edge_id) FILTER (WHERE {IN}) AS n_in_matched,
                                  min(ts) AS first_ts, max(ts) AS last_ts
                           FROM read_parquet('{PAT}') GROUP BY attempt_id""").df()
attempts['touch'] = attempts.n_in >= 1                                            # 평가 구간에 거래가 있음
attempts['inside'] = (attempts.first_ts >= start) & (attempts.last_ts < end)      # 평가 구간 안에서 시작하고 끝남
attempts['eligible'] = (~attempts.pattern_type.isin(EXCLUDE) & (attempts.n_in >= 2) & (attempts.n_in == attempts.n_in_matched))
attempts[attempts.touch].groupby('pattern_type').agg(걸친_시도=('touch', 'sum'), 구간_안에서_끝남=('inside', 'sum'), 평가_대상=('eligible', 'sum'),
                                                      전체_거래_중앙값=('n_rows', 'median'), 구간_안_거래_중앙값=('n_in', 'median'))

In [ ]:
# 거래 → 시도. 한 거래가 두 시도에 들어 있으면 번호가 작은 시도로 둔다
attempt_of = con.execute(f"""SELECT edge_id, min(attempt_id) AS attempt_id, arg_min(pattern_type, attempt_id) AS pattern_type, count(*) AS n,
                                    bool_and({IN}) AS in_window
                             FROM read_parquet('{PAT}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
print('두 시도에 동시에 속한 거래:', int((attempt_of.n > 1).sum()))
attempt_of = attempt_of[~attempt_of.pattern_type.isin(EXCLUDE)]
eval_attempts = set(attempts.loc[attempts.eligible, 'attempt_id'])
truth = attempt_of[attempt_of.attempt_id.isin(eval_attempts) & attempt_of.in_window]      # 정답: 평가할 시도의 구간 안 거래
print(f'평가할 시도 {len(eval_attempts):,} 개 | 정답 거래 {len(truth):,} 건')

In [ ]:
# 모델이 애초에 몇 건을 잡았나: 어떤 묶기 방법도 이것보다 잘할 수는 없다(연결 거래로 되찾는 경우 제외)
seed_ids = set(members.loc[members.seed_model, 'edge_id'])
t = truth.assign(seed=lambda d: d.edge_id.isin(seed_ids))
by_attempt = t.groupby('attempt_id').agg(size=('seed', 'size'), seeds=('seed', 'sum'))
ceiling = dict(시도_수=len(by_attempt), 모델이_전부_잡은_시도=int((by_attempt.seeds == by_attempt['size']).sum()),
               모델이_2건_이상_잡은_시도=int((by_attempt.seeds >= 2).sum()), 모델이_하나도_못_잡은_시도=int((by_attempt.seeds == 0).sum()))
ceiling

## 3. 묶음 만들기
- 묶음은 **거래 2건 이상**만 센다. 생성기는 혼자인 씨앗을 Alert 로 내지 않으므로 단순 연결도 같게 맞춘다.
- 생성기는 제외 패턴의 씨앗도 넣고 돌린 결과에서 그 거래만 뺀다. 단순 연결은 제외 패턴 씨앗을 빼고 잇는다.

In [ ]:
keep = ~members.pattern_type.isin(EXCLUDE)

def as_groups(frame, column):
    g = frame[['edge_id', column, 'y']].rename(columns={column: 'group'})
    return g[g.groupby('group')['edge_id'].transform('size') >= 2]

methods = {'G 생성기': as_groups(members[(members.role > 0) & keep], 'alert')}
seeds = members[members.seed_model & keep].sort_values(['ts', 'edge_id']).reset_index(drop=True)
lonely = {}
for name, hours in BASELINES.items():
    seeds['component'] = GL.link_by_account(seeds, hours)
    methods[name] = as_groups(seeds, 'component')
    lonely[name] = len(seeds) - len(methods[name])
print('씨앗', len(seeds), '| 혼자 남아 묶음이 못 된 씨앗:', lonely)

## 4. 정답과 비교

In [ ]:
summaries, per_attempt = [], {}
for name, groups in methods.items():
    summary, per = GL.compare_to_truth(name, groups, attempt_of, eval_attempts, truth=truth)
    summaries.append(summary)
    per_attempt[name] = per
result = pd.DataFrame(summaries).set_index('방법').T.astype(object)   # 정수는 정수로 보이게
result

**읽는 법**
- `세탁_비율`: 묶인 거래 중 세탁 거래의 비율.
- `시도_2개이상_섞인_묶음`: 서로 다른 세탁 시도가 한 묶음에 들어간 수. 조사자가 한 사건으로 보게 되는 것이 실제로는 여러 사건이다.
- `쌍_정밀도`: 같은 묶음에 든 두 거래가 실제로 같은 시도일 비율. 큰 묶음에 이것저것 섞이면 급격히 떨어진다.
- `쌍_재현율`: 같은 시도의 두 거래가 같은 묶음에 들어갔을 비율.
- `온전히_한_묶음`: 시도의 (구간 안) 거래 전부가 한 묶음에 들어간 시도 수. 2절의 `모델이_전부_잡은_시도` 가 사실상의 상한이다.
- `평균_최대_포함률`: 시도마다 가장 많이 담은 묶음 하나가 그 시도 거래의 몇 %를 담았는지의 평균. 큰 시도는 이 값으로 본다.
- `쪼개진_시도`: 시도의 거래가 묶음 2개 이상에 나뉜 수.

## 5. 패턴별

In [ ]:
frames = []
for name, per in per_attempt.items():
    f = per.groupby('pattern_type').agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                        포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                        쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)))
    frames.append(f.add_prefix(name.split()[0] + '_'))
by_pattern = pd.concat(frames, axis=1)
by_pattern

## 5-1. 시도 크기별
거래 2건짜리 시도와 큰 시도는 묶기 난이도가 다르다. 크기(평가 구간 안 거래 수)로 나눠 본다.

In [ ]:
SIZE_BINS, SIZE_LABELS = [1, 2, 5, 10, 20, 10_000], ['2건', '3~5건', '6~10건', '11~20건', '21건 이상']
frames = []
for name, per in per_attempt.items():
    bucket = pd.cut(per['size'], SIZE_BINS, labels=SIZE_LABELS)
    f = per.groupby(bucket, observed=True).agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                               포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                               쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)))
    frames.append(f.add_prefix(name.split()[0] + '_'))
by_size = pd.concat(frames, axis=1)
by_size

## 6. 큰 묶음 들여다보기
단순 연결은 계좌 하나가 여러 사건을 이어 버리면 묶음이 커진다. 방법마다 가장 큰 묶음 5개를 본다.

In [ ]:
rows = []
for name, groups in methods.items():
    g = groups.merge(attempt_of[['edge_id', 'attempt_id']], on='edge_id', how='left')
    stat = g.groupby('group').agg(거래=('edge_id', 'size'), 세탁=('y', 'sum'), 시도_수=('attempt_id', 'nunique'))
    for group, r in stat.sort_values('거래', ascending=False).head(5).iterrows():
        rows.append(dict(방법=name, 거래=int(r.거래), 세탁=int(r.세탁), 시도_수=int(r.시도_수)))
pd.DataFrame(rows)

## 7. 저장

In [ ]:
result.to_csv(OUT / 'summary.csv')
by_pattern.to_csv(OUT / 'by_pattern.csv')
by_size.to_csv(OUT / 'by_size.csv')
pd.concat([per.assign(method=name) for name, per in per_attempt.items()]).to_csv(OUT / 'per_attempt.csv', index=False)
json.dump(dict(run=RUN, exclude=EXCLUDE, baselines=BASELINES, ceiling=ceiling, lonely_seeds=lonely),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))